# Second-Order Methods — Exercises

8 exercises covering Newton's method, BFGS, Gauss-Newton, natural gradient, and K-FAC.

| Format            | Description                                  |
| ----------------- | -------------------------------------------- |
| **Problem**       | Markdown cell with task description          |
| **Your Solution** | Code cell with scaffolding                   |
| **Solution**      | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus                |
| ----- | --------- | -------------------- |
| ★     | 1-3       | Core mechanics       |
| ★★    | 4-6       | Deeper theory        |
| ★★★   | 7-8       | AI / ML applications |

### Topic Map

| Topic    | Exercise |
| -------- | -------- |
| Newton's method | 1, 2 |
| BFGS update | 3 |
| Gauss-Newton | 4 |
| Natural gradient | 5 |
| L-BFGS | 6 |
| Hessian-vector products | 7 |
| K-FAC approximation | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', expected)
        print('  got     :', got)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def newton_step(x, grad_fn, hess_fn):
    g = grad_fn(x)
    H = hess_fn(x)
    return x - la.solve(H, g)

def bfgs_update(H, s, y):
    ys = y @ s
    if ys < 1e-10:
        return H
    rho = 1.0 / ys
    I = np.eye(len(s))
    V = I - rho * np.outer(s, y)
    return V @ H @ V.T + rho * np.outer(s, s)

print('Setup complete.')

---

### Exercise 1: Newton's Method on a 2D Quadratic ★

Implement Newton's method on $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top A \mathbf{x}$ and verify one-step convergence.

**(a)** For $A = \begin{bmatrix} 3 & 1 \\ 1 & 2 \end{bmatrix}$, compute the Newton step from $\mathbf{x}_0 = (1, 1)$.

**(b)** Verify that $f(\mathbf{x}_1) = 0$ (convergence in one step).

**(c)** Explain why Newton's method converges in one step for quadratic functions.

In [ ]:
# Exercise 1: Your Solution
A = np.array([[3.0, 1.0], [1.0, 2.0]])
x0 = np.array([1.0, 1.0])

def f_quad(x): return 0.5 * x @ A @ x
def grad_quad(x): return A @ x
def hess_quad(x): return A

# Newton step
x1 = newton_step(x0, grad_quad, hess_quad)
print(f'x1 = {x1}')
print(f'f(x1) = {f_quad(x1):.2e}')

In [ ]:
# Exercise 1: Solution
A = np.array([[3.0, 1.0], [1.0, 2.0]])
x0 = np.array([1.0, 1.0])

def f_quad(x): return 0.5 * x @ A @ x
def grad_quad(x): return A @ x
def hess_quad(x): return A

x1 = newton_step(x0, grad_quad, hess_quad)

header('Exercise 1: Newton on a 2D Quadratic')
print(f'x0 = {x0}')
print(f'x1 = {x1}')
print(f'f(x1) = {f_quad(x1):.2e}')

check_close('Newton converges to zero', x1, np.zeros(2), tol=1e-12)
check_close('f(x1) = 0', f_quad(x1), 0.0, tol=1e-12)

print('\nTakeaway: Newton\'s method converges in one step for quadratics because the quadratic model is exact.')

---

### Exercise 2: Quadratic Convergence Verification ★

Verify the quadratic convergence of Newton's method on $f(x) = (x-3)^4 + (x-3)^2$.

**(a)** Compute $f'(x)$ and $f''(x)$ analytically.

**(b)** Run Newton's method from $x_0 = 5$ for 6 iterations.

**(c)** Verify that $|x_{t+1} - x^*| \approx C |x_t - x^*|^2$ (error squares each step).

In [ ]:
# Exercise 2: Your Solution
def f_nc(x): return (x-3)**4 + (x-3)**2
def grad_nc(x): return 4*(x-3)**3 + 2*(x-3)
def hess_nc(x): return 12*(x-3)**2 + 2

x = 5.0
x_star = 3.0
errors = [abs(x - x_star)]

for t in range(6):
    x = x - grad_nc(x) / hess_nc(x)
    errors.append(abs(x - x_star))

print('Quadratic convergence verification:')
for t, err in enumerate(errors):
    print(f'  t={t}: error = {err:.2e}')

# Check ratio of successive error squares
for t in range(2, len(errors)):
    if errors[t-1] > 1e-15:
        ratio = errors[t] / errors[t-1]**2
        print(f'  err[{t}] / err[{t-1}]^2 = {ratio:.4f}')

In [ ]:
# Exercise 2: Solution
def f_nc(x): return (x-3)**4 + (x-3)**2
def grad_nc(x): return 4*(x-3)**3 + 2*(x-3)
def hess_nc(x): return 12*(x-3)**2 + 2

x = 5.0
x_star = 3.0
errors = [abs(x - x_star)]

for t in range(6):
    x = x - grad_nc(x) / hess_nc(x)
    errors.append(abs(x - x_star))

header('Exercise 2: Quadratic Convergence')
for t, err in enumerate(errors):
    print(f'  t={t}: error = {err:.2e}')

# Verify quadratic convergence: ratio should be approximately constant
ratios = []
for t in range(2, len(errors)):
    if errors[t-1] > 1e-15:
        ratios.append(errors[t] / errors[t-1]**2)

if ratios:
    print(f'\nRatios err[t]/err[t-1]^2: {[f"{r:.4f}" for r in ratios]}')
    # Check that ratios are approximately constant (within a factor of 10)
    check_true('ratios are roughly constant', max(ratios)/min(ratios) < 100)

check_close('final error near zero', errors[-1], 0.0, tol=1e-12)

print('\nTakeaway: Quadratic convergence means the number of correct digits doubles each iteration.')

---

### Exercise 3: BFGS Update Computation ★

Compute the BFGS update by hand and verify the secant equation.

**(a)** Given $H_0 = I$, $\mathbf{s} = [1, 2]^\top$, $\mathbf{y} = [3, 1]^\top$, compute $H_1$ using the BFGS formula.

**(b)** Verify that $H_1 \mathbf{y} = \mathbf{s}$ (the secant equation).

**(c)** Verify that $H_1$ is symmetric positive definite.

In [ ]:
# Exercise 3: Your Solution
H0 = np.eye(2)
s = np.array([1.0, 2.0])
y = np.array([3.0, 1.0])

H1 = bfgs_update(H0, s, y)
print(f'H1 = \n{H1}')
print(f'H1 @ y = {H1 @ y}')
print(f's = {s}')
print(f'Eigenvalues of H1: {la.eigvalsh(H1)}')

In [ ]:
# Exercise 3: Solution
H0 = np.eye(2)
s = np.array([1.0, 2.0])
y = np.array([3.0, 1.0])

H1 = bfgs_update(H0, s, y)

header('Exercise 3: BFGS Update')
print(f'H1 = \n{H1}')

check_close('secant equation: H1 @ y = s', H1 @ y, s, tol=1e-10)
check_true('H1 is symmetric', np.allclose(H1, H1.T))
check_true('H1 is positive definite', np.all(la.eigvalsh(H1) > 0))

print('\nTakeaway: BFGS preserves positive definiteness and satisfies the secant equation.')

---

### Exercise 4: Gauss-Newton for Exponential Fitting ★★

Fit $y = a e^{bx}$ to data using the Gauss-Newton method.

**(a)** Implement the residual function $r_i(a, b) = y_i - a e^{bx_i}$ and its Jacobian.

**(b)** Run Gauss-Newton starting from $(a, b) = (1, 1)$.

**(c)** Verify convergence and compare with scipy's least_squares.

In [ ]:
# Exercise 4: Your Solution
x_data = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_data = np.array([1.1, 2.9, 8.1, 22.0, 59.5])

def residuals(params):
    a, b = params
    return y_data - a * np.exp(b * x_data)

def jacobian(params):
    a, b = params
    J = np.zeros((len(x_data), 2))
    # YOUR CODE HERE: fill in J[:, 0] and J[:, 1]
    pass
    return J

params = np.array([1.0, 1.0])
for t in range(20):
    J = jacobian(params)
    r = residuals(params)
    d = -la.solve(J.T @ J + 1e-8*np.eye(2), J.T @ r)
    params = params + d
    if np.linalg.norm(d) < 1e-10:
        break

print(f'Fitted: a={params[0]:.6f}, b={params[1]:.6f}')
print(f'Final residual norm: {np.linalg.norm(residuals(params)):.6f}')

In [ ]:
# Exercise 4: Solution
x_data = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_data = np.array([1.1, 2.9, 8.1, 22.0, 59.5])

def residuals(params):
    a, b = params
    return y_data - a * np.exp(b * x_data)

def jacobian(params):
    a, b = params
    J = np.zeros((len(x_data), 2))
    J[:, 0] = -np.exp(b * x_data)
    J[:, 1] = -a * x_data * np.exp(b * x_data)
    return J

params = np.array([1.0, 1.0])
for t in range(20):
    J = jacobian(params)
    r = residuals(params)
    d = -la.solve(J.T @ J + 1e-8*np.eye(2), J.T @ r)
    params = params + d
    if np.linalg.norm(d) < 1e-10:
        break

header('Exercise 4: Gauss-Newton Curve Fitting')
print(f'Fitted: a={params[0]:.6f}, b={params[1]:.6f}')
res_norm = np.linalg.norm(residuals(params))
print(f'Final residual norm: {res_norm:.6f}')

check_true('residual norm is small', res_norm < 1.0)
check_true('a is positive', params[0] > 0)
check_true('b is positive', params[1] > 0)

print('\nTakeaway: Gauss-Newton exploits the least-squares structure for efficient nonlinear fitting.')

---

### Exercise 5: Natural Gradient for a Univariate Gaussian ★★

Compute the Fisher information matrix for $\mathcal{N}(\mu, \sigma^2)$ and derive the natural gradient.

**(a)** The Fisher matrix for $\mathcal{N}(\mu, \sigma^2)$ is $F = \text{diag}(1/\sigma^2, 1/(2\sigma^4))$. Compute $F^{-1}$.

**(b)** For the negative log-likelihood $\ell(\mu, \sigma^2) = \frac{1}{2}\log(2\pi\sigma^2) + \frac{(x-\mu)^2}{2\sigma^2}$, compute $\nabla \ell$.

**(c)** Compute the natural gradient $F^{-1} \nabla \ell$ for a single observation $x = 5$ with current parameters $\mu = 3, \sigma^2 = 4$.

In [ ]:
# Exercise 5: Your Solution
mu = 3.0
sigma2 = 4.0
x_obs = 5.0

# Fisher matrix
F = np.diag([1.0/sigma2, 1.0/(2*sigma2**2)])
F_inv = np.diag([sigma2, 2*sigma2**2])

# Gradient of negative log-likelihood
dell_dmu = -(x_obs - mu) / sigma2
dell_dsigma2 = 0.5/sigma2 - 0.5*(x_obs - mu)**2 / sigma2**2
grad = np.array([dell_dmu, dell_dsigma2])

# Natural gradient
nat_grad = F_inv @ grad

print(f'Fisher: F = diag({1/sigma2:.4f}, {1/(2*sigma2**2):.4f})')
print(f'Gradient: [{dell_dmu:.4f}, {dell_dsigma2:.4f}]')
print(f'Natural gradient: [{nat_grad[0]:.4f}, {nat_grad[1]:.4f}]')
print(f'Standard gradient: [{grad[0]:.4f}, {grad[1]:.4f}]')

In [ ]:
# Exercise 5: Solution
mu = 3.0
sigma2 = 4.0
x_obs = 5.0

F = np.diag([1.0/sigma2, 1.0/(2*sigma2**2)])
F_inv = np.diag([sigma2, 2*sigma2**2])

dell_dmu = -(x_obs - mu) / sigma2
dell_dsigma2 = 0.5/sigma2 - 0.5*(x_obs - mu)**2 / sigma2**2
grad = np.array([dell_dmu, dell_dsigma2])

nat_grad = F_inv @ grad

header('Exercise 5: Natural Gradient for Gaussian')
print(f'Fisher: F = diag({1/sigma2:.4f}, {1/(2*sigma2**2):.4f})')
print(f'F^{-1} = diag({sigma2:.4f}, {2*sigma2**2:.4f})')
print(f'Gradient: [{dell_dmu:.4f}, {dell_dsigma2:.4f}]')
print(f'Natural gradient: [{nat_grad[0]:.4f}, {nat_grad[1]:.4f}]')

# Verify: natural gradient for mu should be (x - mu)
check_close('natural grad for mu = x - mu', nat_grad[0], x_obs - mu, tol=1e-10)

print('\nTakeaway: Natural gradient simplifies the update by accounting for the statistical geometry.')

---

### Exercise 6: L-BFGS Two-Loop Recursion ★★

Implement the L-BFGS two-loop recursion and verify it matches full BFGS.

**(a)** Implement the two-loop recursion with $m = 2$ history pairs.

**(b)** Compare the search direction with full BFGS for a 3D quadratic.

**(c)** Verify they produce the same direction when $m$ equals the number of iterations.

In [ ]:
# Exercise 6: Your Solution
def two_loop_recursion(g, s_list, y_list, rho_list, gamma):
    q = g.copy()
    m = len(s_list)
    alpha = np.zeros(m)
    
    # Loop 1: backward
    for i in range(m - 1, -1, -1):
        alpha[i] = rho_list[i] * np.dot(s_list[i], q)
        q = q - alpha[i] * y_list[i]
    
    r = gamma * q
    
    # Loop 2: forward
    for i in range(m):
        beta = rho_list[i] * np.dot(y_list[i], r)
        r = r + (alpha[i] - beta) * s_list[i]
    
    return -r

# Test on 3D quadratic
A = np.diag([1.0, 5.0, 10.0])
x0 = np.array([1.0, 1.0, 1.0])

# Run L-BFGS for 3 iterations with m=3
x = x0.copy()
s_list, y_list, rho_list = [], [], []
g = A @ x

for t in range(3):
    gamma = 1.0
    if s_list:
        gamma = np.dot(s_list[-1], y_list[-1]) / np.dot(y_list[-1], y_list[-1])
    
    d_lbfgs = two_loop_recursion(g, s_list, y_list, rho_list, gamma)
    
    # Full BFGS for comparison
    H = np.eye(3)
    for i in range(len(s_list)):
        H = bfgs_update(H, s_list[i], y_list[i])
    d_bfgs = -H @ g
    
    print(f'Iter {t}: ||d_LBFGS - d_BFGS|| = {la.norm(d_lbfgs - d_bfgs):.2e}')
    
    # Take a step
    s = 0.1 * d_lbfgs
    x_new = x + s
    g_new = A @ x_new
    y = g_new - g
    ys = y @ s
    if ys > 1e-10:
        s_list.append(s)
        y_list.append(y)
        rho_list.append(1.0/ys)
    x = x_new
    g = g_new

In [ ]:
# Exercise 6: Solution
def two_loop_recursion(g, s_list, y_list, rho_list, gamma):
    q = g.copy()
    m = len(s_list)
    alpha = np.zeros(m)
    for i in range(m - 1, -1, -1):
        alpha[i] = rho_list[i] * np.dot(s_list[i], q)
        q = q - alpha[i] * y_list[i]
    r = gamma * q
    for i in range(m):
        beta = rho_list[i] * np.dot(y_list[i], r)
        r = r + (alpha[i] - beta) * s_list[i]
    return -r

A = np.diag([1.0, 5.0, 10.0])
x0 = np.array([1.0, 1.0, 1.0])
x = x0.copy()
s_list, y_list, rho_list = [], [], []
g = A @ x

header('Exercise 6: L-BFGS Two-Loop Recursion')
all_match = True
for t in range(3):
    gamma = 1.0
    if s_list:
        gamma = np.dot(s_list[-1], y_list[-1]) / np.dot(y_list[-1], y_list[-1])
    d_lbfgs = two_loop_recursion(g, s_list, y_list, rho_list, gamma)
    H = np.eye(3)
    for i in range(len(s_list)):
        H = bfgs_update(H, s_list[i], y_list[i])
    d_bfgs = -H @ g
    diff = la.norm(d_lbfgs - d_bfgs)
    print(f'Iter {t}: ||d_LBFGS - d_BFGS|| = {diff:.2e}')
    if diff > 1e-10:
        all_match = False
    s = 0.1 * d_lbfgs
    x_new = x + s
    g_new = A @ x_new
    y = g_new - g
    ys = y @ s
    if ys > 1e-10:
        s_list.append(s)
        y_list.append(y)
        rho_list.append(1.0/ys)
    x = x_new
    g = g_new

check_true('L-BFGS matches BFGS with m >= iterations', all_match)

print('\nTakeaway: L-BFGS with m history pairs reproduces full BFGS when m >= number of iterations.')

---

### Exercise 7: Hessian-Vector Products via Finite Differences ★★★

Implement and verify Hessian-vector products without forming the full Hessian.

**(a)** For $f(\mathbf{x}) = \sum_{i=1}^n x_i^4 + \frac{1}{2}\mathbf{x}^\top A \mathbf{x}$, implement the gradient function.

**(b)** Compute $H\mathbf{v}$ using finite differences of the gradient.

**(c)** Compare with the exact Hessian-vector product and verify accuracy.

In [ ]:
# Exercise 7: Your Solution
n = 20
A = np.random.randn(n, n)
A = A.T @ A + np.eye(n)

def f_hvp(x):
    return np.sum(x**4) + 0.5 * x @ A @ x

def grad_f_hvp(x):
    # YOUR CODE HERE
    pass

def hvp_fd(grad_fn, x, v, eps=1e-6):
    # YOUR CODE HERE
    pass

x = np.random.randn(n)
v = np.random.randn(n)

# Exact Hessian-vector product
H_exact = np.diag(12*x**2) + A
Hv_exact = H_exact @ v

# Finite difference approximation
Hv_fd = hvp_fd(grad_f_hvp, x, v)

rel_error = la.norm(Hv_exact - Hv_fd) / la.norm(Hv_exact)
print(f'Relative error: {rel_error:.2e}')

In [ ]:
# Exercise 7: Solution
n = 20
A = np.random.randn(n, n)
A = A.T @ A + np.eye(n)

def f_hvp(x):
    return np.sum(x**4) + 0.5 * x @ A @ x

def grad_f_hvp(x):
    return 4 * x**3 + A @ x

def hvp_fd(grad_fn, x, v, eps=1e-6):
    return (grad_fn(x + eps*v) - grad_fn(x - eps*v)) / (2*eps)

x = np.random.randn(n)
v = np.random.randn(n)

H_exact = np.diag(12*x**2) + A
Hv_exact = H_exact @ v
Hv_fd = hvp_fd(grad_f_hvp, x, v)

header('Exercise 7: Hessian-Vector Products')
rel_error = la.norm(Hv_exact - Hv_fd) / la.norm(Hv_exact)
print(f'Relative error: {rel_error:.2e}')

check_close('HVP via finite differences is accurate', Hv_fd, Hv_exact, tol=1e-3)

print('\nTakeaway: HVP via finite differences avoids forming the full Hessian, enabling Newton-like methods at O(n) cost.')

---

### Exercise 8: K-FAC Approximation Analysis ★★★

Analyze the quality of the K-FAC Kronecker approximation for a linear layer.

**(a)** For a linear layer with input $A \in \mathbb{R}^{n \times d_{in}}$ and output gradients $G \in \mathbb{R}^{n \times d_{out}}$, the exact Fisher is $F = \frac{1}{n} \sum_i \text{vec}(a_i g_i^\top) \text{vec}(a_i g_i^\top)^\top$.

**(b)** The K-FAC approximation is $F \approx A^\top A / n \otimes G^\top G / n$.

**(c)** For small dimensions, compute both and measure the relative error. Analyze when K-FAC is exact.

In [ ]:
# Exercise 8: Your Solution
np.random.seed(42)
d_in, d_out, n = 4, 3, 200

A_data = np.random.randn(n, d_in)
G_data = np.random.randn(n, d_out)

# Exact Fisher (small enough to compute)
F_exact = np.zeros((d_in*d_out, d_in*d_out))
for i in range(n):
    vec_i = np.outer(A_data[i], G_data[i]).ravel()
    F_exact += np.outer(vec_i, vec_i)
F_exact /= n

# K-FAC approximation
A_cov = A_data.T @ A_data / n
G_cov = G_data.T @ G_data / n
F_kfac = np.kron(G_cov, A_cov)

# Measure approximation quality
rel_error = la.norm(F_exact - F_kfac) / la.norm(F_exact)
print(f'd_in={d_in}, d_out={d_out}, n={n}')
print(f'Relative error: {rel_error:.4f}')
print(f'Exact Fisher rank: {np.linalg.matrix_rank(F_exact, tol=1e-8)}')
print(f'K-FAC Fisher rank: {np.linalg.matrix_rank(F_kfac, tol=1e-8)}')

# Check eigenvalue alignment
eig_exact = np.sort(la.eigvalsh(F_exact))[::-1]
eig_kfac = np.sort(la.eigvalsh(F_kfac))[::-1]
print(f'Top 3 eigenvalues (exact): {eig_exact[:3]}')
print(f'Top 3 eigenvalues (K-FAC): {eig_kfac[:3]}')

In [ ]:
# Exercise 8: Solution
np.random.seed(42)
d_in, d_out, n = 4, 3, 200

A_data = np.random.randn(n, d_in)
G_data = np.random.randn(n, d_out)

F_exact = np.zeros((d_in*d_out, d_in*d_out))
for i in range(n):
    vec_i = np.outer(A_data[i], G_data[i]).ravel()
    F_exact += np.outer(vec_i, vec_i)
F_exact /= n

A_cov = A_data.T @ A_data / n
G_cov = G_data.T @ G_data / n
F_kfac = np.kron(G_cov, A_cov)

header('Exercise 8: K-FAC Approximation Analysis')
rel_error = la.norm(F_exact - F_kfac) / la.norm(F_exact)
print(f'd_in={d_in}, d_out={d_out}, n={n}')
print(f'Relative error: {rel_error:.4f}')
print(f'Exact Fisher rank: {np.linalg.matrix_rank(F_exact, tol=1e-8)}')
print(f'K-FAC Fisher rank: {np.linalg.matrix_rank(F_kfac, tol=1e-8)}')

eig_exact = np.sort(la.eigvalsh(F_exact))[::-1]
eig_kfac = np.sort(la.eigvalsh(F_kfac))[::-1]
print(f'\nTop 3 eigenvalues (exact): {eig_exact[:3]}')
print(f'Top 3 eigenvalues (K-FAC): {eig_kfac[:3]}')

check_true('K-FAC captures dominant eigenvalues', eig_kfac[0] / eig_exact[0] > 0.5)
check_true('relative error is reasonable', rel_error < 1.0)

print('\nTakeaway: K-FAC approximates the Fisher with a Kronecker product, reducing storage from O(d_in^2 * d_out^2) to O(d_in^2 + d_out^2).')

---

## What to Review After Finishing

- [ ] Exercise 1: Newton's method converges in one step for quadratics
- [ ] Exercise 2: Quadratic convergence — error squares at each step
- [ ] Exercise 3: BFGS update preserves positive definiteness and satisfies secant equation
- [ ] Exercise 4: Gauss-Newton for nonlinear least squares curve fitting
- [ ] Exercise 5: Natural gradient simplifies updates via statistical geometry
- [ ] Exercise 6: L-BFGS two-loop recursion matches full BFGS with sufficient memory
- [ ] Exercise 7: Hessian-vector products via finite differences
- [ ] Exercise 8: K-FAC Kronecker approximation quality analysis

## References

1. Nocedal, J. & Wright, S. (2006). *Numerical Optimization* (2nd ed.). Springer.
2. Martens, J. & Grosse, R. (2015). "Optimizing neural networks with Kronecker-factored approximate curvature." ICML.
3. Amari, S. (1998). "Natural gradient works efficiently in learning." Neural Computation.